# giskard × Isaac Sim Stretch 控制演示

通过 [giskardpy](https://github.com/cram2/cognitive_robot_abstract_machine) 的闭环 QP 全身控制器，
向仿真中的 Stretch 发送关节、末端笛卡尔、底盘和夹爪运动目标。

**运行前提**：
1. 仿真已启动：`~/.local/bin/isaacsim_python_wrapper.sh examples/apartment.py`
2. giskard 服务端已启动（等日志 `giskard is ready`）：
   `cognitive_robot_abstract_machine/.venv/bin/python giskard_stretch/giskard_stretch_isaac.py`
3. 本 notebook 的内核选择 **Giskard Python (venv)**（Docker 镜像已注册；本机手动注册：
   `.venv/bin/python -m ipykernel install --user --name giskard`）
4. Jupyter 进程的环境里已 source `/opt/ros/jazzy/setup.bash` 和 `ros2_ws/install/setup.bash`


## 连接 giskard

从服务端拉取世界模型，并定义一个小的执行助手。`add_end_conditions` 会在目标达成后
保持 1 秒再结束运动（稳定相）——giskard 的行为树在目标完成与终止归零之间会持续发布
最后一帧速度约 1 秒，不加稳定相底盘会滑过目标。


In [ ]:
import os, sys
sys.path.insert(0, os.getcwd())  # giskard_stretch/

from giskard_client import add_end_conditions, connect
from giskardpy.motion_statechart.data_types import ObservationStateValues
from giskardpy.motion_statechart.motion_statechart import MotionStatechart

giskard = connect('giskard_notebook_client')
world = giskard.world
print('connected, robot:', giskard.robot_name)


def run_goal(task, timeout=60.0):
    """Execute a single motion task with settle phase + timeout."""
    msc = MotionStatechart()
    msc.add_node(task)
    add_end_conditions(msc, task, timeout_seconds=timeout)
    giskard.execute(msc)
    reached = msc.observation_state[task] == ObservationStateValues.TRUE
    print('goal reached:', reached)
    return reached


## 1. 关节空间控制

直接给关节目标位置。可用关节名见 `stretch_joints.CONTROLLED_JOINTS`。
阈值 0.02：本管线的稳态精度在 0.01–0.02 之间，默认的 0.01 会偶发判不达。


In [ ]:
from giskardpy.motion_statechart.tasks.joint_tasks import JointPositionList
from semantic_digital_twin.datastructures.joint_state import JointState

goal = {
    world.get_connection_by_name('joint_lift'): 0.8,
    world.get_connection_by_name('joint_wrist_yaw'): 0.0,
}
run_goal(JointPositionList(goal_state=JointState.from_mapping(goal), threshold=0.02),
         timeout=30.0)


## 2. 夹爪开合


In [ ]:
FINGER_OPEN, FINGER_CLOSED = 0.109, 0.0

def gripper(position):
    goal = {
        world.get_connection_by_name('joint_gripper_finger_left'): position,
        world.get_connection_by_name('joint_gripper_finger_right'): position,
    }
    return run_goal(JointPositionList(goal_state=JointState.from_mapping(goal),
                                      threshold=0.02), timeout=30.0)

gripper(FINGER_OPEN)


In [ ]:
gripper(FINGER_CLOSED)


## 3. 末端笛卡尔位姿（仅手臂）

让 `link_grasp_center` 沿自身坐标系平移，保持当前朝向；`root_link=base_link`
表示只用手臂运动链（底盘不动）。注意：当前没有启用自碰撞规避，
不要把目标设到会顶到机器人自身的位置（如手腕弯曲时向下压）。


In [ ]:
from giskardpy.motion_statechart.tasks.cartesian_tasks import CartesianPose
from semantic_digital_twin.spatial_types.spatial_types import Pose, Vector3

tip = world.get_kinematic_structure_entity_by_name('link_grasp_center')
base = world.get_kinematic_structure_entity_by_name('base_link')

goal_pose = Pose(position=Vector3(0.0, 0.0, 0.15), reference_frame=tip)  # 沿夹爪 z 上移 15cm
run_goal(CartesianPose(root_link=base, tip_link=tip, goal_pose=goal_pose, threshold=0.03))


## 4. 末端笛卡尔位姿（全身）

`root_link=world.root` 时底盘参与运动——目标超出手臂可达范围时底盘会自动移动补足。


In [ ]:
goal_pose = Pose(position=Vector3(0.4, 0.0, 0.0), reference_frame=tip)
run_goal(CartesianPose(root_link=world.root, tip_link=tip, goal_pose=goal_pose,
                       threshold=0.03), timeout=90.0)


## 5. 底盘移动

`DifferentialDriveBaseGoal`（先转向 → 直行 → 转到目标朝向）是差速底盘的正确用法；
直接用 `CartesianPose` 约束底盘位姿会在目标附近打摆。阈值 5cm 是移动底盘的常规容差。


In [ ]:
from giskardpy.motion_statechart.goals.cartesian_goals import DifferentialDriveBaseGoal

goal_pose = Pose(position=Vector3(0.5, 0.0, 0.0), reference_frame=base)  # 底盘前方 0.5m
run_goal(DifferentialDriveBaseGoal(goal_pose=goal_pose, threshold=0.05))


## 故障排查

- **夹爪接触物体后卡住**：用仿真原生接口复位：
  `ros2 topic pub --once /stretch/gripper_command std_msgs/msg/Float64 "{data: 0.05}"`
- **手臂姿态拧了**：用 `/stretch/joint_command`（JointState）直接摆回标准姿态
- giskard 是全身控制器，仅手臂的目标也可能让其他受控关节被 QP 轻微调整
- 更多说明见 `giskard_stretch/README.md`
